[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/duoan/TorchCode/blob/master/templates/11_sliding_window.ipynb)

# 🔴 Hard: Sliding Window Attention

Implement **Sliding Window Attention** — used in Longformer, Mistral, etc. for efficient long-context processing.

Each position $i$ can only attend to positions $j$ where $|i - j| \le w$ (the window size).

### Signature
```python
def sliding_window_attention(Q, K, V, window_size):
    # Q, K, V: (batch, seq, d) → output: (batch, seq, d_v)
    # window_size: int — position i attends to [i-w, i+w]
```

### Rules
- Do **NOT** use sparse attention libraries
- Mask positions outside the window with `-inf`
- `window_size=0`: only self — output should equal V
- `window_size >= seq_len`: equivalent to full attention

In [24]:
# Install torch-judge in Colab (no-op in JupyterLab/Docker)
try:
    import google.colab
    get_ipython().run_line_magic('pip', 'install -q torch-judge')
except ImportError:
    pass


# 0 can attend to 0,1,2
# 1 can attend to 0,1,2,3
# 2 can attend to 0,1,2,3,4
# 3 can attejd t0 1,2,3,4,5

In [25]:
import torch
import math

In [26]:
mask = torch.zeros(3, 4)
for i in range(3):
    for j in range(4):
        if abs(j-i) > 4:
            mask[i][j] = 1
mask

tensor([[0., 0., 0., 0.],
        [0., 0., 0., 0.],
        [0., 0., 0., 0.]])

Torch, torch.unsqueeze() is used to add a dimension of size one at a specific position in a tensor

In [33]:
torch.arange(3).unsqueeze(1)    # (seq_q,1)


tensor([[0],
        [1],
        [2]])

In [34]:
torch.arange(4).unsqueeze(0)  

tensor([[0, 1, 2, 3]])

In [ ]:
# ✏️ YOUR IMPLEMENTATION HERE

def sliding_window_attention(Q, K, V, window_size):
    d_k = Q.shape[2]
    seq_len_q = Q.shape[1]
    seq_len_k = K.shape[1]
    scores = torch.bmm(Q, torch.transpose(K, 1, 2))/math.sqrt(d_k)
    # mask = torch.zeros(seq_len_q, seq_len_k)
    # for i in range(seq_len_q):
    #     for j in range(seq_len_k):
    #         if abs(j-i) > window_size:
    #             mask[i][j] = 1
    idx_q = torch.arange(seq_len_q, device=Q.device).unsqueeze(1)    # (seq_q,1)
    idx_k = torch.arange(seq_len_k, device=Q.device).unsqueeze(0)    # (1,seq_k)
    # mask = (idx_k - idx_q).abs() > window_size                      # (seq_q, seq_k), bool
    scores.masked_fill_(mask.bool(), -float('inf'))
    return torch.bmm(torch.softmax(scores, dim=-1), V)


    pass  # Replace this

In [31]:
# 🧪 Debug
Q = torch.randn(1, 6, 8)
K = torch.randn(1, 6, 8)
V = torch.randn(1, 6, 8)

out = sliding_window_attention(Q, K, V, window_size=1)
print("Output shape:", out.shape)  # (1, 6, 8)

# window=0 should return V
out0 = sliding_window_attention(Q, K, V, window_size=0)
print("window=0 == V?", torch.allclose(out0, V, atol=1e-5))

Output shape: torch.Size([1, 6, 8])
window=0 == V? True


In [32]:
from torch_judge import check
check('sliding_window')


🧪 Testing: Sliding Window Attention (Hard)
──────────────────────────────────────────────────
  ✅ [1/5] Output shape (1.0ms)
  ✅ [2/5] window_size=0 — only sees itself (1.1ms)
  ✅ [3/5] Large window equals full attention (3.9ms)
  ✅ [4/5] Distant tokens don't affect output (4.7ms)
  ✅ [5/5] Gradient flow (1.0ms)
──────────────────────────────────────────────────
  🎉 All 5 tests passed! (11.8ms total)
  Progress saved. Run status() to see your dashboard.

